In [2]:
import pandas as pd
import numpy as np

import json
with open('config.json', 'r') as f:
    config = json.load(f)
    
from helpers.sementic_recoding import drop_by_threshold,recode_semantic_missingness
from helpers.modeling import (
    prepare_data,
    identify_column_types,
    create_preprocessor,
    evaluate_model,
    run_grid_search,
)
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR, NuSVR  # regression
from sklearn.neighbors import KNeighborsRegressor

df = pd.read_csv(config['filepath'], encoding="latin-1", header=[0, 1])
print(f"Initial shape: {df.shape}")

Initial shape: (2189, 92)


In [3]:
df.columns = df.columns.get_level_values(1)

column_names = config['column_names']
df.columns = column_names

df = df.iloc[1:].reset_index(drop=True)
df = df.drop(columns=['mix_id', 'country', 'reference_details', 'publish_year'])

for col in df.columns:
    converted = pd.to_numeric(df[col], errors='coerce')
    if converted.isna().sum() == df[col].isna().sum():
        df[col] = converted

print(f"Cleaned shape: {df.shape}")
print(f"Total columns: {len(df.columns)}")
df['paper_reference'] = df['paper_reference'].ffill()
df.head()

Cleaned shape: (2188, 88)
Total columns: 88


,cement,cement_type,cement_grade,silica_fume,fly_ash,fly_ash_type,limestone_powder,quartz_powder,glass_powder,rice_husk_ash,...,shrinkage_standard,shrinkage_28d,shrinkage_56d,freeze_thaw_standard,freeze_thaw_cycles,rcpt_standard,rcpt,resistivity_standard,surface_resistivity,paper_reference
0,839.0,Type I/II low-alkali portland cement,NaN,104.0,104.0,Class-F,0.0,0.0,0.0,0.0,...,ASTM C157,270.0,371.0,ASTM C666,105.0,ASTM C1202,165.0,AASHTO T 358,386.0,Ref-1-data
1,839.0,Type I/II low-alkali portland cement,NaN,104.0,52.0,Class-F,0.0,0.0,0.0,0.0,...,ASTM C157,242.0,317.0,ASTM C666,106.0,ASTM C1202,142.0,AASHTO T 358,424.0,Ref-1-data
2,839.0,Type I/II low-alkali portland cement,NaN,104.0,26.0,Class-F,0.0,0.0,0.0,0.0,...,ASTM C157,223.0,290.0,ASTM C666,106.0,ASTM C1202,136.0,AASHTO T 358,463.0,Ref-1-data
3,839.0,Type I/II low-alkali portland cement,NaN,104.0,0.0,NaN,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ref-1-data
4,839.0,Type I/II low-alkali portland cement,NaN,104.0,52.0,Class-F,0.0,0.0,0.0,0.0,...,ASTM C157,231.0,286.0,ASTM C666,106.0,ASTM C1202,135.0,AASHTO T 358,437.0,Ref-1-data


In [4]:
# Check data types
print("Data types summary:")
print(df.dtypes.value_counts())
print(f"\nMissing values:\n{df.isnull().sum().sum()} total missing values")

Data types summary:
float64    57
str        31
Name: count, dtype: int64

Missing values:
111322 total missing values


In [5]:
# Remove rows with missing target variable
df = df.dropna(subset=['cs_28d'])
print(f"Rows with 28-day compressive strength data: {len(df)}")

# Define columns to drop (focusing on cs_28d as target)
drop_cols = config["drop_cols"]

# Drop unnecessary columns
df = df.drop(columns=drop_cols)
print(f"Final shape after filtering: {df.shape}")
print(f"Remaining columns: {df.shape[1]}")



Rows with 28-day compressive strength data: 2073
Final shape after filtering: (2073, 44)
Remaining columns: 44


In [6]:
df_cleaned = pd.read_csv(config["initial_cleaned_filepath"], index_col=0)
df_cleaned['paper_reference'] = df['paper_reference'].values
df_cleaned.iloc[1]
print(f"Final shape after filtering: {df.shape}")
print(f"Remaining columns: {df.shape[1]}")
df_cleaned.head(20)

Final shape after filtering: (2073, 44)
Remaining columns: 44


,cement,cement_type,cement_grade,silica_fume,fly_ash,fly_ash_type,limestone_powder,quartz_powder,glass_powder,rice_husk_ash,...,fiber2_youngs_modulus,water,sp_type,sp_amount,curing_method,curing_temp,curing_humidity,curing_pressure,cs_28d,paper_reference
0,839.0,OPC_I,NaN,104.0,104.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,56.50,Standard Curing,NaN,NaN,NaN,135.0,Ref-1-data
1,839.0,OPC_I,NaN,104.0,52.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,59.33,Standard Curing,NaN,NaN,NaN,132.0,Ref-1-data
2,839.0,OPC_I,NaN,104.0,26.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,59.33,Standard Curing,NaN,NaN,NaN,122.5,Ref-1-data
3,839.0,OPC_I,NaN,104.0,0.0,NaN,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,62.15,Standard Curing,NaN,NaN,NaN,116.0,Ref-1-data
4,839.0,OPC_I,NaN,104.0,52.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,64.98,Standard Curing,NaN,NaN,NaN,134.0,Ref-1-data
5,839.0,OPC_I,NaN,104.0,26.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,64.98,Standard Curing,NaN,NaN,NaN,131.5,Ref-1-data
6,839.0,OPC_I,NaN,104.0,0.0,NaN,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,67.80,Standard Curing,NaN,NaN,NaN,130.5,Ref-1-data
7,839.0,OPC_I,NaN,104.0,83.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,59.33,Standard Curing,NaN,NaN,NaN,134.5,Ref-1-data
8,839.0,OPC_I,NaN,104.0,62.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,62.15,Standard Curing,NaN,NaN,NaN,132.5,Ref-1-data
9,839.0,OPC_I,NaN,104.0,0.0,NaN,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,64.98,Standard Curing,NaN,NaN,NaN,123.0,Ref-1-data


In [7]:
df_cleaned.dtypes

cement                     float64
cement_type                    str
cement_grade               float64
silica_fume                float64
fly_ash                    float64
fly_ash_type                   str
limestone_powder           float64
quartz_powder              float64
glass_powder               float64
rice_husk_ash              float64
metakaolin                 float64
ggbfs                      float64
slag                       float64
slag_type                      str
nano_caco3                 float64
nano_al2o3                 float64
nano_tio2                  float64
nano_sio2                  float64
filler                     float64
filler_type                    str
sand                       float64
sand_type                      str
sand_max_size              float64
fiber1_type                    str
fiber1_amount              float64
fiber1_length              float64
fiber1_diameter            float64
fiber1_tensile_strength    float64
fiber1_youngs_modulu

In [8]:
## Semantic Missingness Recoding

# Define material amount/type pairs for semantic analysis
amount_type_twins = config["amount_type_twins"]

# Apply semantic recoding
df_semantic_recoded = df_cleaned.copy()

for amount_col, type_col in amount_type_twins:
    before_amount = df[amount_col].isna().sum()
    before_type = df[type_col].isna().sum()
    
    recode_semantic_missingness(df_semantic_recoded, amount_col, type_col)
    
    print(f"\n{amount_col} / {type_col}")
    print(f"  NaN amount:      {before_amount} → {df[amount_col].isna().sum()}  (true reporting gap)")
    print(f"  NaN type:        {before_type} → {df[type_col].isna().sum()}  (true reporting gap)")
    print(f"  unknown_type:    {(df[type_col] == 'unknown_type').sum()}  (recoded)")
    print(f"  not_applicable:  {(df[type_col] == 'not_applicable').sum()}  (recoded)")
    
# Recoded + 50% missing data threshold
df_semantic_recod_50 = df_semantic_recoded.copy()
df_semantic_recod_50 = drop_by_threshold(df_semantic_recod_50, 0.5)
print(f"Recoded + 50% threshold shape: {df_semantic_recod_50.shape}")


fly_ash / fly_ash_type
  NaN amount:      0 → 0  (true reporting gap)
  NaN type:        1647 → 1647  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

slag / slag_type
  NaN amount:      0 → 0  (true reporting gap)
  NaN type:        1981 → 1981  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

filler / filler_type
  NaN amount:      0 → 0  (true reporting gap)
  NaN type:        1713 → 1713  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

sand / sand_type
  NaN amount:      0 → 0  (true reporting gap)
  NaN type:        82 → 82  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

fiber1_amount / fiber1_type
  NaN amount:      669 → 669  (true reporting gap)
  NaN type:        669 → 669  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

fiber2_amount / fiber2_type
  NaN amount:      1952 → 1952  (true repo

In [9]:
df_semantic_recod_50 = df_semantic_recod_50.drop(columns=['cement_grade']) # cement grade 38% missing, encodes sames info as cement type
fiber_cols = ['fiber1_length', 'fiber1_diameter']  
df_semantic_recod_50[fiber_cols] = df_semantic_recod_50[fiber_cols].fillna(0)  # replacing with mean wrong
df_semantic_recod_50.head()

df_semantic_recod_50.to_csv("../Datasets/processed/UHPC_dataset/semantic_recoding_features_50_with_publications.csv", index=False)

In [ ]:
df = df_semantic_recod_50
target_col = 'cs_28d'

X_train, X_val, X_test, y_train, y_val, y_test = prepare_data(df, target_col)

print(f"Train set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

X = df.drop(columns=[target_col])
numerical_cols, one_hot_columns, k_fold_columns = identify_column_types(X)

preprocessor = create_preprocessor( numerical_cols, one_hot_columns, k_fold_columns, 
                                   handle_unknown='ignore')

Train set size: 1451
Validation set size: 311
Test set size: 311


In [10]:
# KNN PIPELINE CREATION
knn_pipeline = Pipeline([('preprocessor', preprocessor), ('model', KNeighborsRegressor())])

knn_grid = {
    'model__n_neighbors': list(range(1, 31)),
    'model__weights': ['uniform', 'distance'],
    'model__p': [1, 2, 3],
    'model__metric': ['euclidean', 'manhattan', 'minkowski']
}

gs_knn = run_grid_search(knn_pipeline, knn_grid, X_train, X_val, X_test, y_train, y_val, y_test, 'KNN')


GridSearchCV will test 540 combinations for KNN...

BEST HYPERPARAMETERS — KNN
  metric: manhattan
  n_neighbors: 8
  p: 1
  weights: distance
Best CV RMSE: 22.1019


c:\Users\shoai\anaconda3\envs\aicome\Lib\site-packages\threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)



VALIDATION SET PERFORMANCE
RMSE: 19.6037
MAE: 14.3755
R2: 0.7300
Correlation: 0.8549

TEST SET PERFORMANCE
RMSE: 20.3166
MAE: 14.9601
R2: 0.6471
Correlation: 0.8059

RESULTS SUMMARY
          Metric  Validation      Test
           RMSE   19.603709 20.316566
            MAE   14.375529 14.960077
             R2    0.729980  0.647089
Correlation (R)    0.854910  0.805927

TOP 10 COMBINATIONS
   metric  n_neighbors  p  weights  mean_test_score   CV_rmse
manhattan            8  1 distance      -488.494149 22.101904
minkowski            7  1 distance      -490.483974 22.146873
minkowski           13  1 distance      -491.072883 22.160164
manhattan           10  1 distance      -491.883636 22.178450
minkowski           16  1 distance      -493.073161 22.205251
minkowski            9  1 distance      -494.018439 22.226526
manhattan           11  3 distance      -494.075588 22.227811
manhattan           17  1 distance      -494.220352 22.231067
manhattan           14  1 distance      -494.25

In [11]:
# SVR PIPELINE CREATION
svr_pipeline = Pipeline([('preprocessor', preprocessor), ('model', SVR(kernel='rbf'))])

svr_grid = {
    'model__C':        [128, 256, 512, 1024],
    'model__epsilon': [0.01, 0.1, 0.5, 1, 3]
}

gs_svr = run_grid_search(svr_pipeline, svr_grid, X_train, X_val, X_test, y_train, y_val, y_test, 'SVR')


GridSearchCV will test 20 combinations for SVR...

BEST HYPERPARAMETERS — SVR
  C: 1024
  epsilon: 3
Best CV RMSE: 22.6338

VALIDATION SET PERFORMANCE
RMSE: 21.1448
MAE: 15.9580
R2: 0.6859
Correlation: 0.8285

TEST SET PERFORMANCE
RMSE: 21.2916
MAE: 15.9193
R2: 0.6124
Correlation: 0.7832

RESULTS SUMMARY
          Metric  Validation      Test
           RMSE   21.144825 21.291551
            MAE   15.958031 15.919347
             R2    0.685857  0.612405
Correlation (R)    0.828496  0.783217

TOP 10 COMBINATIONS
   C  epsilon  mean_test_score   CV_rmse
1024      3.0      -512.289389 22.633811
1024      0.5      -515.442170 22.703352
 512      3.0      -515.770524 22.710582
 128      3.0      -516.688879 22.730791
 512      0.5      -517.506499 22.748769
1024      0.1      -518.827088 22.777776
 512      1.0      -519.053319 22.782742
 128      0.5      -519.109018 22.783964
 256      0.5      -520.119633 22.806131
1024      1.0      -520.305004 22.810195


In [12]:
# NuSVR PIPELINE CREATION
nusvr_pipeline = Pipeline([('preprocessor', preprocessor), ('model', NuSVR(kernel='rbf'))])

nusvr_grid = {
    'model__C':   [128, 256, 512, 1024],
    'model__nu': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
}

gs_nusvr = run_grid_search(nusvr_pipeline, nusvr_grid, X_train, X_val, X_test, y_train, y_val, y_test, 'NuSVR')


GridSearchCV will test 36 combinations for NuSVR...

BEST HYPERPARAMETERS — NuSVR
  C: 512
  nu: 0.4
Best CV RMSE: 22.5309

VALIDATION SET PERFORMANCE
RMSE: 21.3939
MAE: 16.1737
R2: 0.6784
Correlation: 0.8239

TEST SET PERFORMANCE
RMSE: 22.0374
MAE: 16.7997
R2: 0.5848
Correlation: 0.7655

RESULTS SUMMARY
          Metric  Validation      Test
           RMSE   21.393922 22.037378
            MAE   16.173736 16.799659
             R2    0.678411  0.584775
Correlation (R)    0.823876  0.765507

TOP 10 COMBINATIONS
   C  nu  mean_test_score   CV_rmse
 512 0.4      -507.641279 22.530896
1024 0.5      -509.876615 22.580448
 256 0.5      -509.984439 22.582835
 256 0.6      -510.039083 22.584045
1024 0.7      -510.099657 22.585386
 128 0.4      -510.191001 22.587408
1024 0.4      -510.476433 22.593726
 128 0.5      -511.130945 22.608205
 512 0.5      -511.643551 22.619539
 512 0.6      -513.967047 22.670841


In [25]:
best_params_knn = gs_knn.best_params_
best_params_svr = gs_svr.best_params_
best_params_nusvr = gs_nusvr.best_params_


with open('results.json', 'r') as f:
    results = json.load(f)

results["best_params"] = results.get("best_params", {})
results["best_params"]["best_params_publications_included"] = {
    "knn": best_params_knn,
    "svr": best_params_svr,
    "nusvr": best_params_nusvr,
}

with open('results.json', "w") as f:
    json.dump(results, f, indent=2)

print(results["best_params"])


{'best_params_publications_included': {'knn': {'model__metric': 'manhattan', 'model__n_neighbors': 8, 'model__p': 1, 'model__weights': 'distance'}, 'svr': {'model__C': 1024, 'model__epsilon': 3}, 'nusvr': {'model__C': 512, 'model__nu': 0.4}}, 'recoded_50': {'knn': {'model__metric': 'minkowski', 'model__n_neighbors': 3, 'model__p': 1, 'model__weights': 'distance'}, 'svr': {'model__C': 1024, 'model__epsilon': 3, 'model__gamma': 'scale'}, 'nusvr': {'model__C': 1024, 'model__gamma': 'scale', 'model__nu': 0.4}}}
